# Stage 0 Zero-Shot B (T4)

Thin Colab orchestration notebook for zero-shot `f04_candidate_yes_no` and `f05_candidate_yes_no_full_context` using the existing repo scripts and configs.


## 1. Mount Google Drive

Mount Google Drive before running the same Colab bootstrap flow used in `starter_notebook.ipynb`.


In [17]:
from google.colab import drive

drive.mount("/content/drive")


: 

## 2. Configure Paths

Set the repo checkout path, Drive output directory, and optional data override here. The setup cell below uses the same upload-widget bootstrap code as `starter_notebook.ipynb`.

If the upload widget appears, select your local `.env` file and rerun the setup cell.


In [18]:
from pathlib import Path

REPO_URL = "https://github.com/Demetri65/dl-kaggle-competition-final.git"
REPO_REF = "main"
REPO_DIR = Path("/content/dl-kaggle-competition-final")  # Change this if you want the repo checkout elsewhere
SOURCE_ENV = Path("/content/.env")
UPLOADER_KEY = "_env_uploader"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/p2p_runs/notebook_b_t4"
DATA_DIR_OVERRIDE = ""  # Optional: set this to an existing dataset directory for the run commands


## 3. Prepare The Repo

This is the same Colab environment setup flow used in `starter_notebook.ipynb`: upload `.env`, sync the repo, then run `scripts/bootstrap_colab.sh`.


In [19]:
# ── 0. Colab setup ───────────────────────────────────────────────
import subprocess
import ipywidgets as widgets
from IPython.display import display

def get_uploaded_file(uploader):
    value = uploader.value

    if isinstance(value, dict):
        filename, uploaded_file = next(iter(value.items()))
        if isinstance(uploaded_file, dict):
            content = uploaded_file.get("content", uploaded_file.get("data"))
        else:
            content = uploaded_file
    else:
        uploaded_file = value[0]
        if isinstance(uploaded_file, dict):
            filename = uploaded_file["name"]
            content = uploaded_file["content"]
        else:
            filename = uploaded_file.name
            content = uploaded_file.content

    payload = content.tobytes() if hasattr(content, "tobytes") else bytes(content)
    return filename, payload

ready_to_bootstrap = SOURCE_ENV.exists()

if ready_to_bootstrap:
    print(f"Using existing {SOURCE_ENV}")
else:
    uploader = globals().get(UPLOADER_KEY)
    if uploader is None:
        uploader = widgets.FileUpload(accept=".env", multiple=False, description="Upload .env")
        globals()[UPLOADER_KEY] = uploader

    if not uploader.value:
        display(uploader)
        print("Select your local .env file in the upload widget above, then rerun this cell.")
    else:
        filename, payload = get_uploaded_file(uploader)
        SOURCE_ENV.write_bytes(payload)
        SOURCE_ENV.chmod(0o600)
        print(f"Saved {filename} to {SOURCE_ENV}")
        uploader.close()
        globals().pop(UPLOADER_KEY, None)
        ready_to_bootstrap = True

if ready_to_bootstrap:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_REF], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "FETCH_HEAD"], check=True)

    print(f"Repo synced to latest origin/{REPO_REF} at {REPO_DIR}")
    result = subprocess.run(
        ["bash", "scripts/bootstrap_colab.sh"],
        cwd=REPO_DIR,
        capture_output=True,
        text=True,
    )
    if result.stdout:
        print(result.stdout, end="")
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr, end="")
        raise RuntimeError(f"scripts/bootstrap_colab.sh failed with exit code {result.returncode}")


## 4. Helpers

These helpers keep the runnable cells thin and route jobs through the existing repo scripts.


In [20]:
import os
import shlex
import subprocess

REPO_ROOT = REPO_DIR.resolve()
Path(DRIVE_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
os.chdir(REPO_ROOT)

COMMON_OVERRIDES = []
if DATA_DIR_OVERRIDE:
    COMMON_OVERRIDES.append(f"data.data_dir={DATA_DIR_OVERRIDE}")

def with_common_overrides(overrides):
    return [*COMMON_OVERRIDES, *overrides]

def extend_with_overrides(args, overrides):
    for override in overrides:
        args.extend(["--set", override])

def run_repo_command(args):
    command = ["python3", *args]
    print("$", " ".join(shlex.quote(part) for part in command))
    result = subprocess.run(command, cwd=REPO_ROOT, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="")
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result

print(f"Repo root: {REPO_ROOT}")
print(f"Drive output dir: {DRIVE_OUTPUT_DIR}")
if DATA_DIR_OVERRIDE:
    print(f"Data override: {DATA_DIR_OVERRIDE}")


## 5. Smoke Test

Optional smoke test on a small validation subset for `f04_candidate_yes_no` and `f05_candidate_yes_no_full_context`.


In [21]:
smoke_args = [
    "scripts/run_stage.py",
    "f04_candidate_yes_no",
    "f05_candidate_yes_no_full_context",
    "--output-dir", f"{DRIVE_OUTPUT_DIR}/smoke",
]
extend_with_overrides(smoke_args, with_common_overrides([
    "training.epochs=0",
    "lora.enabled=false",
    "training.bf16=false",
    "training.fp16=true",
    "training.eval_batch_size=4",
    "runtime.max_val_examples=32",
]))
run_repo_command(smoke_args)


## 6. Full Run

Run the full zero-shot `f04_candidate_yes_no` and `f05_candidate_yes_no_full_context` sweep on T4.


In [22]:
full_args = [
    "scripts/run_stage.py",
    "f04_candidate_yes_no",
    "f05_candidate_yes_no_full_context",
    "--output-dir", f"{DRIVE_OUTPUT_DIR}/full",
]
extend_with_overrides(full_args, with_common_overrides([
    "training.epochs=0",
    "lora.enabled=false",
    "training.bf16=false",
    "training.fp16=true",
    "training.eval_batch_size=8",
]))
run_repo_command(full_args)


## 7. Summarize Results

Print the top validation runs from `results/experiments.csv`.


In [ ]:
summary_args = [
    "scripts/summarize_results.py",
    "--sort-by", "val_accuracy",
    "--top", "20",
]
run_repo_command(summary_args)
